In [ ]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import emcee
from find_source import summary
import corner
import math
from astropy.coordinates import Angle
import astropy.units as units
import scipy.special as sp
import warnings
import itertools

In [714]:
def p_model(p_params, u, v):
    i0, l0, m0 = p_params
    return i0 * np.exp(-2*np.pi*1j*(u*l0 + v*m0))

def c_model(c_params, u, v):
    s0, l0, m0, vis_sigma = c_params
    return s0 * np.exp(-0.5*(u**2 + v**2)/vis_sigma**2) * np.exp(-2*np.pi*1j*(u*l0 + v*m0))

def g_model(g_params, u, v):
    s0, l0, m0, vis_sigma, ratio, vis_theta = g_params
    return s0 * np.exp(-0.5*((u*np.cos(vis_theta)-v*np.sin(vis_theta))**2 + (u*np.sin(vis_theta)+v*np.cos(vis_theta))**2/ratio**2)/vis_sigma**2) \
            * np.exp(-2*np.pi*1j*(u*l0 + v*m0))

def d_model(d_params, u, v):
    s0, l0, m0, vis_r = d_params
    return s0 * 2*vis_r/np.sqrt(u**2+v**2) * sp.j1(np.sqrt(u**2+v**2)/vis_r) * np.exp(-2*np.pi*1j*(u*l0 + v*m0))

In [715]:
def p_p0(peak, rad_coord, rad_pix, rad_barea, total_flux, nwalkers):
    p0 = np.zeros((nwalkers, 3))
    for i in range(nwalkers):
        p0[i,0] = np.random.uniform(0.95*peak, 1.05*peak)
        p0[i,1] = np.random.uniform(-rad_pix/2+rad_coord[0], rad_pix/2+rad_coord[0])
        p0[i,2] = np.random.uniform(-rad_pix/2+rad_coord[1], rad_pix/2+rad_coord[1])
    return p0

def c_p0(peak, rad_coord, rad_pix, rad_barea, total_flux, nwalkers):
    p0 = np.zeros((nwalkers, 4))
    source_area = rad_barea * total_flux / peak
    sigma = np.sqrt(source_area / (2*np.pi))
    vis_sigma = 1/(2*np.pi*sigma)
    for i in range(nwalkers):
        p0[i,0] = np.random.uniform(0.95*peak, 1.05*peak)
        p0[i,1] = np.random.uniform(-rad_pix/2+rad_coord[0], rad_pix/2+rad_coord[0])
        p0[i,2] = np.random.uniform(-rad_pix/2+rad_coord[1], rad_pix/2+rad_coord[1])
        p0[i,3] = np.random.uniform(0.95*vis_sigma, 1.05*vis_sigma)
    return p0

def g_p0(peak, rad_coord, rad_pix, rad_barea, total_flux, nwalkers):
    p0 = np.zeros((nwalkers, 6))
    source_area = rad_barea * total_flux / peak
    sigma = np.sqrt(source_area / (2*np.pi))
    vis_sigma = 1/(2*np.pi*sigma)
    for i in range(nwalkers):
        p0[i,0] = np.random.uniform(0.95*peak, 1.05*peak)
        p0[i,1] = np.random.uniform(-rad_pix/2+rad_coord[0], rad_pix/2+rad_coord[0])
        p0[i,2] = np.random.uniform(-rad_pix/2+rad_coord[1], rad_pix/2+rad_coord[1])
        p0[i,3] = np.random.uniform(0.95*vis_sigma, 1.05*vis_sigma)
        p0[i,4] = np.random.uniform(0, 1)
        p0[i,5] = np.random.uniform(-np.pi/2, np.pi/2)
    return p0

def d_p0(peak, rad_coord, rad_pix, rad_barea, total_flux, nwalkers):
    p0 = np.zeros((nwalkers, 4))
    source_area = rad_barea * total_flux / peak
    r = np.sqrt(source_area / (2*np.pi))
    vis_r = 1/(math.pi*r)
    for i in range(nwalkers):
        p0[i,0] = np.random.uniform(0.95*peak, 1.05*peak)
        p0[i,1] = np.random.uniform(-rad_pix/2+rad_coord[0], rad_pix/2+rad_coord[0])
        p0[i,2] = np.random.uniform(-rad_pix/2+rad_coord[1], rad_pix/2+rad_coord[1])
        p0[i,3] = np.random.uniform(0.95*vis_r, 1.05*vis_r)
    return p0

In [716]:
letters = ['c','a','b']
numbers = [1,2,3]
other = ['hello', 'world', '!']
both = list(zip(numbers, letters))
both.sort(reverse=True)
three = list(zip(other, both))
three.sort()
three

[('!', (1, 'c')), ('hello', (3, 'b')), ('world', (2, 'a'))]

In [717]:
def p_prior(params):
    i0, l0, m0 = params
    if 0 < i0:
        return 0.0
    return -np.inf

def c_prior(params):
    s0, l0, m0, vis_sigma = params
    if 0 < s0 and 0 < vis_sigma:
        return 0.0
    return -np.inf

def g_prior(params):
    s0, l0, m0, vis_sigma, ratio, vis_theta = params
    if 0 < s0 and 0 < vis_sigma and 0 < ratio <= 1 and -np.pi/2 <= vis_theta <= np.pi/2:
        return 0.0
    return -np.inf

def d_prior(params):
    s0, l0, m0, vis_r = params
    if 0 < s0 and 0 < vis_r:
        return 0.0
    return -np.inf

In [718]:
def log_likelihood(model, re, im, u, v, w):
    return -0.5 * np.sum(w * ((re - model.real)**2 + (im - model.imag)**2))

In [719]:
SOURCE_TYPES = {'p': [3, p_p0, p_prior, p_model], \
                'c': [4, c_p0, c_prior, c_model], \
                'g': [6, g_p0, g_prior, g_model], \
                'd': [4, d_p0, d_prior, d_model]} # TODO: finish filling in

In [720]:
def log_probability(sources, params, re, im, u, v, w):
    log_prior = 0.0
    model = 0.0
    i = 0
    for source in sources:
        n_params, p0_func, prior_func, model_func = SOURCE_TYPES[source]
        source_params = params[i]
        lp = prior_func(source_params)
        if not np.isfinite(lp):
            return -np.inf
        log_prior += lp
        model += model_func(source_params, u, v)
        i += 1
    log_likelihood_value = log_likelihood(model, re, im, u, v, w)
    return log_prior + log_likelihood_value

In [ ]:
def uv_fit(fits_file, sources, corner_plot=True):

    # Check input source types
    for source in sources:
        if source not in SOURCE_TYPES or source != 'any':
            raise ValueError(f"Source type '{source}' is not recognized. Source type must be one of the following: \
                            'p' (point), 'c' (circular gaussian), 'g' (gaussian), 'd' (disk), or 'any' (try all and pick best fit).")

    # Extract data from fits file
    file = fits.open(fits_file)
    cdelt1 = file[0].header['CDELT1']
    cunit1 = file[0].header['CUNIT1']
    naxis1 = file[0].header['NAXIS1']
    data = file[1].data

    summ = summary(fits_file, plot=False)
    coord0 = summ['int_peak_coord'][0] # relative arcsec
    rad_coord = (float(Angle(coord0[0], units.arcsec).to(units.radian).value), float(Angle(coord0[1], units.arcsec).to(units.radian).value))
    bmaj = file[0].header['BMAJ'] # cunit1
    bmin = file[0].header['BMIN'] # cunit1
    rad_bmaj = Angle(bmaj, cunit1).to(units.radian).value
    rad_bmin = Angle(bmin, cunit1).to(units.radian).value
    rad_barea = np.pi * rad_bmaj * rad_bmin / (4 * np.log(2))
    total_flux = np.max((re**2 + im**2)**(1/2))
    rad_pix = float(Angle(cdelt1, cunit1).to(units.radian).value)
    int_peaks = summ['int_peak_val']
    int_coords = summ['int_peak_coord']
    ext_peaks = summ['ext_peak_val']
    ext_coords = summ['ext_peak_coord']

    int_info = list(zip(int_peaks, int_coords))
    ext_info = list(zip(ext_peaks, ext_coords))
    all_peaks = int_info + ext_info
    all_peaks.sort(reverse=True) # sort by peak value

    vis = np.array(data)
    freq_bin, u, v, re, im, w = [], [], [], [], [], []
    for row in vis:
        freq_bin_data, u_data, v_data, re_data, im_data, w_data = row
        freq_bin.append(int(freq_bin_data))
        u.append(int(u_data))
        v.append(int(v_data))
        re.append(float(re_data/w_data))
        im.append(float(im_data/w_data))
        w.append(float(w_data))

    # Adding in conjugate half of data
    freq_bin *= 2
    neg_u = [-1 * val for val in u]
    u += neg_u
    neg_v = [-1 * val for val in v]
    v += neg_v
    re *= 2
    neg_im = [-1 * val for val in im]
    im += neg_im
    w *= 2

    freq_bin = np.array(freq_bin)
    u = np.array(u)
    v = np.array(v)
    re = np.array(re)
    im = np.array(im)
    w = np.array(w)

    file.close() # good practice

    # All possible permutations
    n_sources = len(sources)
    sample_space = list(SOURCE_TYPES.keys()) * n_sources


    # Initial guesses
    n_walkers = 0
